# im gonna preen !!!
Initial stuff for BERTopic

### Initial Python Data Filtering for converting the data into Spark

Choose a data path below if you want to and put data into spark with filtering

In [15]:
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
  SparkSession.builder
  .appName("Phase2")
  .config("spark.driver.memory", "8g")
  .config("spark.executor.memory", "8g")
  .config("spark.sql.debug.maxToStringFields", "100")
  .getOrCreate()
)

review_df = spark.read.parquet("/home/cs179g/repo/data/clean_reviews.parquet")
row_count = review_df.count()
print("Number of rows: " + str(row_count))

Number of rows: 1787844


In [16]:
small_df = review_df.sample(withReplacement=False, fraction=0.15, seed=3)
row_count = small_df.count()
print("Number of rows: " + str(row_count))

Number of rows: 267876


Compile an array of reviews for the BERTopic model

In [ ]:
from pyspark.sql.functions import collect_list
from pyspark.sql.functions import lit
reviewArr=(clean_df.groupBy(lit(1)).agg(collect_list('review'))).first()['collect_list(review)']

ERROR:root:KeyboardInterrupt while sending command.                 (3 + 2) / 5]
Traceback (most recent call last):
  File "/home/cs179g/.local/lib/python3.10/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/home/cs179g/.local/lib/python3.10/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/home/cs179g/miniconda3/envs/spark_env/lib/python3.10/socket.py", line 717, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 

### Initialize bertopic stuff

In [6]:
from bertopic import BERTopic
from bertopic import __init__
from sklearn.feature_extraction.text import CountVectorizer


vectorizer_model = CountVectorizer(stop_words="english")
topic_model = BERTopic(vectorizer_model=vectorizer_model)
topics, probs = topic_model.fit_transform(reviewArr)

/home/cs179g/miniconda3/envs/spark_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5161.33it/s]


KeyboardInterrupt: 

In [ ]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,0,433,0_game_campaign_like_multiplayer,"[game, campaign, like, multiplayer, just, good...",[Modern Warfare 4 has a lot going against it. ...
1,1,73,1_game_bios_update_play,"[game, bios, update, play, just, boot, tpm, pc...","[UDPATE: GARBAGE Anticheat system, and they wo..."


In [ ]:
embedding_model = "sentence-transformers/all-MiniLM-L6-v2"
topic_model.save("/home/cs179g/repo/data/BERTopicModel", serialization="safetensors", save_ctfidf=True, save_embedding_model=embedding_model)

In [ ]:
#topic_model.transform(reviewArr[300])
representative_docs = topic_model.get_representative_docs()
# print(representative_docs[1])
topic_model.transform(representative_docs[1][1])


([np.int64(1)], array([0.4811409]))